In [10]:
# ============================================================
# FILE 10 — FINAL RAW INFERENCE & EARTH ENGINE ASSET EXPORT
# ------------------------------------------------------------
# PAN-INDIA SOIL HEALTH MAPPING
#
# PURPOSE
# ─────────────────────────────────────────────────────────────
# Generate AEZ-wise RAW soil-health prediction rasters for:
#   N, P, K, OC
#
# using the already-trained and locked Random Forest models.
#
# FINAL FIXES INCLUDED
# ─────────────────────────────────────────────────────────────
# ✅ No LULC masking during inference
#    -> Produces raw prediction layers for all inferable pixels.
#
# ✅ Explicit output CRS
#    -> EPSG:4326.
#
# ✅ Explicit common export grid using crsTransform
#    -> All 72 AEZ-level assets align on the same 30 m grid.
#
# ✅ Explicit per-source-band reprojection/resampling before classification
#    -> Landsat, MODIS, CHIRPS, SRTM, and SoilGrids predictors are
#       all forced to the same final EPSG:4326 30 m grid before stacking.
#
# ✅ No nodata fill values inserted
#    -> Masked pixels remain masked.
#
# ✅ Correct pH band-name handling
#    -> pH_0-5 / pH_5-15 are safely normalized for GEE classifier use.
#
# ✅ Existing asset skip logic
#    -> Safe to rerun without resubmitting completed exports.
#
# ✅ Robust error handling and task summary
#
# IMPORTANT
# ─────────────────────────────────────────────────────────────
# • Models are NOT retrained or modified.
# • This exports RAW layers only.
# • Any LULC masking should be applied later by production users.
# • SoilGrids NoData pixels are NOT artificially filled.
# • Pixels where required source predictors are genuinely missing
#   will remain masked / NoData.
# ============================================================

import os
import re
import glob
import time
import tempfile
import traceback

import ee
import yaml
import joblib

from geemap import ml


# ============================================================
# 1. LOAD CONFIGURATION
# ============================================================

with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

PROJECT_ID = config["gee"]["project_id"]

START_DATE = config["parameters"]["start_date"]
END_DATE = config["parameters"]["end_date"]

INPUT_FOLDER = config["paths"]["models_joblib_dir"]
OUTPUT_FOLDER = config["paths"]["models_csv_dir"]

AEZ_ASSET_PATH = config["gee"]["aez_asset_path"]


# ============================================================
# 2. EXPORT SETTINGS
# ============================================================

MAX_PIXELS = 1e13
SUBMISSION_DELAY = 2

EXPORT_CRS = "EPSG:4326"

# Use a new suffix so existing "_new" assets are not skipped.
# If you want to use the same "_new" names, delete old assets first
# or change this back to "_new".
ASSET_SUFFIX = "_new_reproj"

# Global ~30 m EPSG:4326 grid.
# Same grid used in the previously validated export.
EPSG4326_30M_TRANSFORM = [
    0.000269494585235856472,
    0,
    -180.0,
    0,
    -0.000269494585235856472,
    90.0
]


# ============================================================
# 3. INITIALIZE EARTH ENGINE
# ============================================================

print("\nInitializing Earth Engine...")

# try:
#     ee.Initialize(
#         project=PROJECT_ID,
#         opt_url="https://earthengine-highvolume.googleapis.com"
#     )
#     print("✅ Earth Engine initialized successfully")

# except Exception:
ee.Authenticate(force=True)
ee.Initialize(
    project=PROJECT_ID,
    opt_url="https://earthengine-highvolume.googleapis.com"
)
print("✅ Earth Engine authenticated and initialized")


# ============================================================
# 3A. TARGET PROJECTION FOR ALL SOURCE BANDS
# ------------------------------------------------------------
# IMPORTANT:
# This must come AFTER ee.Initialize().
# Every predictor source is explicitly aligned to this projection
# before stacking/classification.
# ============================================================

TARGET_PROJECTION = ee.Projection(
    EXPORT_CRS,
    EPSG4326_30M_TRANSFORM
)


def align_to_target_grid(image):
    """
    Explicitly resample/reproject a continuous predictor image to
    the final EPSG:4326 30 m grid before classification.

    This does NOT fill missing pixels. It only ensures that each
    input band is on the same grid before being stacked.

    Continuous predictors are resampled using bilinear interpolation.
    """
    return (
        image
        .resample("bilinear")
        .reproject(TARGET_PROJECTION)
        .toFloat()
    )


# ============================================================
# 4. MODEL FILENAME PATTERN
# ============================================================

# Expected format:
# rfr_model_YYYY-MM-DD_AEZ_<N>_<VAR>.joblib
FILE_PATTERN = re.compile(
    r"rfr_model_(.+)_AEZ_(\d+)_(.+)\.joblib$"
)


# ============================================================
# 5. CSV → GEE CLASSIFIER
# ============================================================

def csv_to_classifier_safe(csv_path: str):
    """
    Convert geemap RF-tree CSV to a GEE classifier.

    The sklearn feature names contain:
        pH_0-5
        pH_5-15

    These are normalized to:
        pH_0_5
        pH_5_15

    because hyphens are problematic in the classifier tree text.
    """

    with open(csv_path, "r", encoding="utf-8") as fh:
        csv_text = fh.read()

    csv_text = (
        csv_text
        .replace("pH_0-5", "pH_0_5")
        .replace("pH_5-15", "pH_5_15")
    )

    fd, tmp_path = tempfile.mkstemp(
        suffix="_fixed.csv",
        dir=os.path.dirname(csv_path)
    )
    os.close(fd)

    try:
        with open(tmp_path, "w", encoding="utf-8") as fh:
            fh.write(csv_text)

        classifier = ml.csv_to_classifier(tmp_path)

    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass

    return classifier


# ============================================================
# 6. LANDSAT PREPROCESSING
# ============================================================

def scale_landsat(image):
    """
    Apply Landsat Collection 2 Level-2 surface reflectance scaling
    and mask cloud, cloud shadow, cirrus, and snow/ice pixels.
    """

    qa = image.select("QA_PIXEL")

    mask = (
        qa.bitwiseAnd(1 << 3).eq(0)        # cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # cloud shadow
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # cirrus
        .And(qa.bitwiseAnd(1 << 5).eq(0))  # snow / ice
    )

    scaled = (
        image
        .select(
            [
                "SR_B2",
                "SR_B3",
                "SR_B4",
                "SR_B5",
                "SR_B6",
                "SR_B7"
            ]
        )
        .multiply(0.0000275)
        .add(-0.2)
        .rename(
            [
                "BLUE",
                "GREEN",
                "RED",
                "NIR",
                "SWIR1",
                "SWIR2"
            ]
        )
    )

    return (
        image
        .addBands(scaled)
        .updateMask(mask)
        .copyProperties(image, ["system:time_start"])
    )


def add_seasonal_indices(image):
    """
    Add NDVI and NIRv to Landsat images.
    """

    ndvi = (
        image
        .normalizedDifference(["NIR", "RED"])
        .rename("NDVI")
    )

    nirv = (
        ndvi
        .multiply(image.select("NIR"))
        .rename("NIRv")
    )

    return image.addBands([ndvi, nirv])


def add_derived_indices(image):
    """
    Add annual spectral indices:
        SI, RI, TGSI
    """

    si = (
        image
        .normalizedDifference(["RED", "BLUE"])
        .rename("SI")
    )

    ri = (
        image
        .expression(
            "RED**2 / (BLUE * GREEN**3)",
            {
                "RED": image.select("RED"),
                "BLUE": image.select("BLUE"),
                "GREEN": image.select("GREEN")
            }
        )
        .rename("RI")
    )

    tgsi = (
        image
        .normalizedDifference(["SWIR1", "NIR"])
        .rename("TGSI")
    )

    return image.addBands([si, ri, tgsi])


# ============================================================
# 7. SOILGRIDS LOADER
# ============================================================

SOIL_BAND_MAPS = {
    "sand": {
        "sand05": "sand_0-5cm_mean",
        "sand515": "sand_5-15cm_mean"
    },

    "silt": {
        "silt05": "silt_0-5cm_mean",
        "silt515": "silt_5-15cm_mean"
    },

    "clay": {
        "clay05": "clay_0-5cm_mean",
        "clay515": "clay_5-15cm_mean"
    },

    "phh2o": {
        "pH_0-5": "phh2o_0-5cm_mean",
        "pH_5-15": "phh2o_5-15cm_mean"
    }
}


def load_soil_image(property_name: str, features: list):
    """
    Load only the SoilGrids bands required by the current model.

    pH bands are renamed from:
        pH_0-5   -> pH_0_5
        pH_5-15  -> pH_5_15

    to match the GEE classifier-safe band names.
    """

    band_map = SOIL_BAND_MAPS[property_name]

    selected_bands = []
    renamed_bands = []

    for model_feat, ee_band in band_map.items():
        normalized_feat = model_feat.replace("-", "_")

        if model_feat in features or normalized_feat in features:
            selected_bands.append(ee_band)
            renamed_bands.append(normalized_feat)

    if not selected_bands:
        return None

    img = (
        ee.Image(f"projects/soilgrids-isric/{property_name}_mean")
        .select(selected_bands)
        .rename(renamed_bands)
    )

    if property_name == "phh2o":
        img = img.divide(10.0)

    return img


# ============================================================
# 8. LOAD AEZ FEATURE COLLECTION
# ============================================================

aez_fc = ee.FeatureCollection(AEZ_ASSET_PATH)


# ============================================================
# 9. DISCOVER MODEL FILES
# ============================================================

joblib_files = sorted(
    glob.glob(
        os.path.join(INPUT_FOLDER, "*.joblib")
    )
)

print(f"\nFound {len(joblib_files)} model file(s)")

if not joblib_files:
    raise FileNotFoundError(
        f"No .joblib files found in: {INPUT_FOLDER}"
    )


# ============================================================
# 10. SUMMARY COUNTERS
# ============================================================

submitted_count = 0
skipped_count = 0
failed_count = 0


# ============================================================
# 11. MAIN EXPORT LOOP
# ============================================================

for file_path in joblib_files:

    filename = os.path.basename(file_path)

    print("\n" + "=" * 70)
    print(f"Processing: {filename}")
    print("=" * 70)

    try:

        # ----------------------------------------------------
        # 11.1 Parse model metadata from filename
        # ----------------------------------------------------

        match = FILE_PATTERN.match(filename)

        if not match:
            print("❌ Filename does not match expected pattern — skipping")
            failed_count += 1
            continue

        model_date, aez_str, pred_var = match.groups()

        AEZ = int(aez_str)
        date_compact = model_date.replace("-", "")

        ASSET_NAME = (
            f"AEZ_{AEZ}_{pred_var}_{date_compact}{ASSET_SUFFIX}"
        )

        ASSET_ID = (
            f"projects/ee-mtpictd-ratinder-mcs/assets/{ASSET_NAME}"
        )

        print(f"AEZ        : {AEZ}")
        print(f"Nutrient   : {pred_var}")
        print(f"Asset Name : {ASSET_NAME}")

        # ----------------------------------------------------
        # 11.2 Skip asset if it already exists
        # ----------------------------------------------------

        try:
            ee.data.getAsset(ASSET_ID)

            print("⏭️ Asset already exists — skipping")

            skipped_count += 1
            continue

        except Exception:
            pass

        # ----------------------------------------------------
        # 11.3 Load local sklearn RF model
        # ----------------------------------------------------

        rf = joblib.load(file_path)

        features = list(rf.feature_names_in_)

        n_estimators = rf.get_params()["n_estimators"]
        max_depth = rf.get_params()["max_depth"]

        print(f"Features ({len(features)}):")
        print(features)

        # ----------------------------------------------------
        # 11.4 Locate matching geemap tree CSV
        # ----------------------------------------------------

        csv_filename = (
            f"rf_trees_"
            f"t{n_estimators}_"
            f"d{max_depth}_"
            f"{model_date}_"
            f"AEZ_{AEZ}_"
            f"{pred_var}.csv"
        )

        csv_path = os.path.join(
            OUTPUT_FOLDER,
            csv_filename
        )

        if not os.path.exists(csv_path):
            print("❌ Required CSV not found:")
            print(csv_filename)

            failed_count += 1
            continue

        # ----------------------------------------------------
        # 11.5 Build GEE classifier
        # ----------------------------------------------------

        classifier = csv_to_classifier_safe(csv_path)

        print("✅ Classifier created")

        # ----------------------------------------------------
        # 11.6 Define AEZ region
        # ----------------------------------------------------

        roi = (
            aez_fc
            .filter(ee.Filter.eq("ae_regcode", AEZ))
            .geometry()
        )

        # ----------------------------------------------------
        # 11.7 Build Landsat collection
        # ----------------------------------------------------

        print("Building Landsat composites...")

        LS = (
            ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
            .merge(
                ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
            )
            .filterBounds(roi)
            .filterDate(START_DATE, END_DATE)
            .map(scale_landsat)
        )

        images_to_stack = []

        # ----------------------------------------------------
        # 11.8 Seasonal NDVI / NIRv features
        # ----------------------------------------------------

        LS_seasonal = LS.map(add_seasonal_indices)

        seasons = {
            "Kharif": ("2023-07-01", "2023-10-31"),
            "Rabi": ("2023-11-01", "2024-03-31"),
            "Zaid": ("2024-04-01", "2024-06-30")
        }

        for band in ["NIRv", "NDVI"]:
            for season_name, (s_date, e_date) in seasons.items():

                feat_name = f"{band}_{season_name}"

                if feat_name not in features:
                    continue

                img = (
                    LS_seasonal
                    .filterDate(s_date, e_date)
                    .select(band)
                    .median()
                    .unmask(0)
                    .rename(feat_name)
                )

                # Explicit source-band alignment to final 30 m grid.
                images_to_stack.append(
                    align_to_target_grid(img)
                )

        # ----------------------------------------------------
        # 11.9 Annual spectral indices: SI, RI, TGSI
        # ----------------------------------------------------

        annual_mean = (
            LS
            .select(
                [
                    "BLUE",
                    "GREEN",
                    "RED",
                    "NIR",
                    "SWIR1",
                    "SWIR2"
                ]
            )
            .mean()
        )

        annual_indices = add_derived_indices(annual_mean)

        needed_annual = [
            b for b in ["SI", "RI", "TGSI"]
            if b in features
        ]

        if needed_annual:
            img = (
                annual_indices
                .select(needed_annual)
            )

            # Explicit source-band alignment to final 30 m grid.
            images_to_stack.append(
                align_to_target_grid(img)
            )

        # ----------------------------------------------------
        # 11.10 Temperature
        # ----------------------------------------------------

        if "temp" in features:

            temp = (
                ee.ImageCollection("MODIS/061/MOD11A2")
                .select("LST_Day_1km")
                .filterDate(START_DATE, END_DATE)
                .filterBounds(roi)
                .mean()
                .multiply(0.02)
                .subtract(273.15)
                .rename("temp")
            )

            # Explicit source-band alignment to final 30 m grid.
            images_to_stack.append(
                align_to_target_grid(temp)
            )

        # ----------------------------------------------------
        # 11.11 Precipitation
        # ----------------------------------------------------

        if "precipitation" in features:

            precipitation = (
                ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
                .filterDate(START_DATE, END_DATE)
                .filterBounds(roi)
                .sum()
                .rename("precipitation")
            )

            # Explicit source-band alignment to final 30 m grid.
            images_to_stack.append(
                align_to_target_grid(precipitation)
            )

        # ----------------------------------------------------
        # 11.12 Topography
        # ----------------------------------------------------

        elevation = (
            ee.Image("USGS/SRTMGL1_003")
            .select("elevation")
        )

        if "elevation" in features:
            # Explicit source-band alignment to final 30 m grid.
            images_to_stack.append(
                align_to_target_grid(elevation.rename("elevation"))
            )

        if "slope" in features:
            slope = (
                ee.Terrain.slope(elevation)
                .rename("slope")
            )

            # Explicit source-band alignment to final 30 m grid.
            images_to_stack.append(
                align_to_target_grid(slope)
            )

        # ----------------------------------------------------
        # 11.13 SoilGrids bands
        # ----------------------------------------------------

        for prop in ["sand", "silt", "clay", "phh2o"]:

            soil_img = load_soil_image(
                prop,
                features
            )

            if soil_img is not None:
                # Explicit source-band alignment to final 30 m grid.
                # Note: This does NOT fill SoilGrids NoData pixels.
                images_to_stack.append(
                    align_to_target_grid(soil_img)
                )

        # ----------------------------------------------------
        # 11.14 Build final feature stack
        # ----------------------------------------------------

        if not images_to_stack:
            raise ValueError(
                "No images were added to the feature stack"
            )

        final_stack = (
            ee.Image.cat(images_to_stack)
            .toFloat()
        )

        # ----------------------------------------------------
        # 11.15 Validate required model bands
        # ----------------------------------------------------

        stack_bands = final_stack.bandNames().getInfo()

        normalized_features = [
            f.replace("pH_0-5", "pH_0_5")
             .replace("pH_5-15", "pH_5_15")
            for f in features
        ]

        missing_features = [
            f for f in normalized_features
            if f not in stack_bands
        ]

        if missing_features:
            print("❌ Missing features in stack:")
            print(missing_features)

            failed_count += 1
            continue

        print("✅ Feature stack complete")
        print("✅ All source bands explicitly aligned to final 30 m grid")

        # ----------------------------------------------------
        # 11.16 Run inference
        # ----------------------------------------------------

        print("Running inference...")

        classified = (
            final_stack
            .classify(classifier)
            .rename(pred_var)
            .toFloat()
        )

        # ----------------------------------------------------
        # 11.17 RAW output
        # No LULC mask. No nodata fill.
        # ----------------------------------------------------

        final_soil_map = (
            classified
            .clip(roi)
        )

        # ----------------------------------------------------
        # 11.18 Export to GEE asset
        # Explicit export CRS + common global 30 m transform.
        # ----------------------------------------------------

        print(f"🚀 Submitting export → {ASSET_ID}")

        task = ee.batch.Export.image.toAsset(
            image=final_soil_map,
            description=f"Export_{ASSET_NAME}",
            assetId=ASSET_ID,
            region=roi,
            crs=EXPORT_CRS,
            crsTransform=EPSG4326_30M_TRANSFORM,
            maxPixels=MAX_PIXELS
        )

        task.start()

        print("✅ Export task submitted")

        submitted_count += 1

        time.sleep(SUBMISSION_DELAY)

    except Exception as exc:

        print("\n❌ FAILED")
        print(str(exc))

        traceback.print_exc()

        failed_count += 1


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("BATCH INFERENCE COMPLETE")
print("=" * 70)

print(f"✅ Submitted : {submitted_count}")
print(f"⏭️ Skipped   : {skipped_count}")
print(f"❌ Failed    : {failed_count}")

print("\nMonitor export tasks:")
print("https://code.earthengine.google.com/tasks")

print("=" * 70)


Initializing Earth Engine...


Enter verification code:  4/1AdkVLPy2d5OZ13vrzUDyfr1hGwgSiiHEHkQtPX6vjfeFxOUvL9v7BEwFMnk



Successfully saved authorization token.
✅ Earth Engine authenticated and initialized

Found 72 model file(s)

Processing: rfr_model_2026-03-11_AEZ_10_K.joblib
AEZ        : 10
Nutrient   : K
Asset Name : AEZ_10_K_20260311_new_reproj
⏭️ Asset already exists — skipping

Processing: rfr_model_2026-03-11_AEZ_10_N.joblib
AEZ        : 10
Nutrient   : N
Asset Name : AEZ_10_N_20260311_new_reproj
⏭️ Asset already exists — skipping

Processing: rfr_model_2026-03-11_AEZ_10_OC.joblib
AEZ        : 10
Nutrient   : OC
Asset Name : AEZ_10_OC_20260311_new_reproj
⏭️ Asset already exists — skipping

Processing: rfr_model_2026-03-11_AEZ_10_P.joblib
AEZ        : 10
Nutrient   : P
Asset Name : AEZ_10_P_20260311_new_reproj
⏭️ Asset already exists — skipping

Processing: rfr_model_2026-03-11_AEZ_11_K.joblib
AEZ        : 11
Nutrient   : K
Asset Name : AEZ_11_K_20260311_new_reproj
⏭️ Asset already exists — skipping

Processing: rfr_model_2026-03-11_AEZ_11_N.joblib
AEZ        : 11
Nutrient   : N
Asset Name : AEZ